In [1]:
import re
import math
import numpy as np
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
 
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
 

True

### Get Sentences using Ingestion tool

In [2]:
from components import *

sentences = ingest_using_url("https://jamesclear.com/saying-no")
print(sentences)

combined = get_scores(sentences)
print(combined)
print(f'Number of scores: {len(combined['combined'])}')

c:\Users\Justin\Documents\2026 projects\python\textSummariserExample\summ\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1542.80it/s]


['The Ultimate Productivity Hack is Saying No - James Clear The Ultimate Productivity Hack is Saying No written by James Clear Decision Making Focus Life Lessons The ultimate productivity hack is saying no.', 'Not doing something will always be faster than doing it.', 'This statement reminds me of the old computer programming saying, “Remember that there is no code faster than no code.” 1 The same philosophy applies in other areas of life.', 'For example, there is no meeting that goes faster than not having a meeting at all.', "This is not to say you should never attend another meeting, but the truth is that we say yes to many things we don't actually want to do.", "There are many meetings held that don't need to be held.", 'There is a lot of code written that could be deleted.', "How often do people ask you to do something and you just reply, “Sure thing.” Three days later, you're overwhelmed by how much is on your to-do list.", 'We become frustrated by our obligations even though we 

#### TextRank and centroid similarity are powerful but incomplete.
#### They only look at word overlap and graph structure.
#### They ignore things a human reader would naturally use:

####   - "The first sentence is probably important" (position)
####   - "This sentence mentions the key topic words" (keyword overlap)
####   - "This sentence is too short to be meaningful" (length)

#### Feature engineering adds these human intuitions as numerical signals,
#### then blends them all together with tunable weights.

### 1. Position Score

In [3]:
# 3a. POSITION SCORE

def position_scores(sentences: list[str]) -> np.ndarray:
    """
    Score each sentence by its position in the document.
 
    Returns
    -------
    scores : np.ndarray of shape (n,), normalised to [0, 1]
    """
    n = len(sentences)
    if n == 0:
        return np.array([])
    if n == 1:
        return np.array([1.0])
 
    scores = []
    for i in range(n):
        # relative = 0.0 at first sentence, 1.0 at last sentence
        relative = i / (n - 1)
 
        if relative <= 0.5:
            # First half: score decreases from 1.0 → 0.5
            score = 1.0 - relative
        else:
            # Second half: slight uptick at the end (conclusion)
            score = 0.3 + 0.3 * (1.0 - relative)
 
        scores.append(score)
 
    scores = np.array(scores)
 
    # Normalise to [0, 1]
    s_min, s_max = scores.min(), scores.max()
    if s_max > s_min:
        scores = (scores - s_min) / (s_max - s_min)
 
    return scores

In [4]:
print("\n── 3a. Position Scores (U-shaped curve) ──")
pos = position_scores(sentences)
for i, (s, sc) in enumerate(zip(sentences, pos)):
    bar = "█" * int(sc * 20)
    print(f"  S{i+1}  {sc:.3f}  {bar:<20}  {s[:50]}...")


── 3a. Position Scores (U-shaped curve) ──
  S1  1.000  ████████████████████  The Ultimate Productivity Hack is Saying No - Jame...
  S2  0.985  ███████████████████   Not doing something will always be faster than doi...
  S3  0.969  ███████████████████   This statement reminds me of the old computer prog...
  S4  0.954  ███████████████████   For example, there is no meeting that goes faster ...
  S5  0.939  ██████████████████    This is not to say you should never attend another...
  S6  0.923  ██████████████████    There are many meetings held that don't need to be...
  S7  0.908  ██████████████████    There is a lot of code written that could be delet...
  S8  0.892  █████████████████     How often do people ask you to do something and yo...
  S9  0.877  █████████████████     We become frustrated by our obligations even thoug...
  S10  0.862  █████████████████     2 It's worth asking if things are necessary....
  S11  0.846  ████████████████      Many of them are not, and a simple 

### 2. Keyword / entity overlap score

In [5]:
# 3b. KEYWORD / ENTITY OVERLAP SCORE

def extract_keywords(text: str, top_n: int = 15) -> set[str]:
    """
    Extract the top N most frequent meaningful words from the full document.
 
    Parameters
    ----------
    text  : the full document as a single string
    top_n : how many keywords to extract
 
    Returns
    -------
    keywords : set of keyword strings (lowercased)
    """
    stop_words = set(stopwords.words('english'))
 
    # Tokenise and clean
    tokens = word_tokenize(text.lower())
    tokens = [
        t for t in tokens
        if t.isalpha()           # letters only (no punctuation, numbers)
        and t not in stop_words  # remove stop words
        and len(t) > 3           # skip very short words like 'war', 'big'
    ]
 
    # Count and take top N
    freq = Counter(tokens)
    keywords = {word for word, _ in freq.most_common(top_n)}
    return keywords
 
 
def keyword_overlap_scores(sentences: list[str],
                            keywords: set[str]) -> np.ndarray:
    """
    Score each sentence by the fraction of document keywords it contains.
 
    Score = (keywords found in sentence) / (total keywords)
 
    A sentence mentioning 8 out of 15 keywords scores 8/15 = 0.53
 
    Returns
    -------
    scores : np.ndarray of shape (n,), normalised to [0, 1]
    """
    if not sentences or not keywords:
        return np.zeros(len(sentences))
 
    scores = []
    for sent in sentences:
        # Tokenise the sentence into a set of unique words
        tokens = set(word_tokenize(sent.lower()))
        # Count how many document keywords appear in this sentence
        overlap = len(tokens & keywords)
        scores.append(overlap / len(keywords))
 
    scores = np.array(scores, dtype=float)
 
    # Normalise to [0, 1]
    s_max = scores.max()
    if s_max > 0:
        scores = scores / s_max
 
    return scores

In [6]:
print("\n── 3b. Extracted Keywords (top 15) ──")
full_text = " ".join(sentences)
keywords  = extract_keywords(full_text, top_n=15)
print(f"  {sorted(keywords)}")


── 3b. Extracted Keywords (top 15) ──
  ['also', 'future', 'important', 'life', 'like', 'many', 'need', 'often', 'people', 'productivity', 'saying', 'something', 'things', 'time', 'want']


### 3. Length penalty score

In [7]:
# 3c. LENGTH PENALTY SCORE

def length_penalty_scores(sentences: list[str],
                           ideal_min: int = 10,
                           ideal_max: int = 35) -> np.ndarray:
    """
    Score sentences by how close their word count is to the ideal range.
 
    Parameters
    ----------
    ideal_min : minimum ideal word count (below this = penalised)
    ideal_max : maximum ideal word count (above this = penalised)
 
    Returns
    -------
    scores : np.ndarray of shape (n,), values in [0, 1]
 
    Examples
    --------
    5 words   → 5/10 = 0.50  (too short)
    15 words  → 1.00          (ideal)
    30 words  → 1.00          (ideal)
    50 words  → ~0.72         (a bit long)
    100 words → ~0.52         (too long)
    """
    scores = []
    for sent in sentences:
        wc = len(sent.split())
 
        if ideal_min <= wc <= ideal_max:
            score = 1.0
 
        elif wc < ideal_min:
            # Linear scale up from 0 to 1 as we approach ideal_min
            score = wc / ideal_min
 
        else:
            # Logarithmic penalty for sentences longer than ideal
            # log1p(0) = 0 → no penalty at ideal_max
            # log1p(65) ≈ 4.2 → heavy penalty at 100 words
            excess = wc - ideal_max
            score = max(0.0, 1.0 - 0.05 * math.log1p(excess))
 
        scores.append(score)
 
    return np.array(scores)

In [ ]:
print("\n── 3b. Keyword Overlap Scores ──")
kw = keyword_overlap_scores(sentences, keywords)
for i, (s, sc) in enumerate(zip(sentences, kw)):
    bar = "█" * int(sc * 20)
    print(f"  S{i+1}  {sc:.3f}  {bar:<20}  {s[:50]}...")


── 3b. Keyword Overlap Scores ──
  S1  0.500  ██████████            The Ultimate Productivity Hack is Saying No - Jame...
  S2  0.167  ███                   Not doing something will always be faster than doi...
  S3  0.333  ██████                This statement reminds me of the old computer prog...
  S4  0.000                        For example, there is no meeting that goes faster ...
  S5  0.500  ██████████            This is not to say you should never attend another...
  S6  0.333  ██████                There are many meetings held that don't need to be...
  S7  0.000                        There is a lot of code written that could be delet...
  S8  0.500  ██████████            How often do people ask you to do something and yo...
  S9  0.000                        We become frustrated by our obligations even thoug...
  S10  0.167  ███                   2 It's worth asking if things are necessary....
  S11  0.167  ███                   Many of them are not, and a simple “no” will 

### 4. Domain presets

In [ ]:
# 3d. DOMAIN WEIGHT PRESETS

DOMAIN_WEIGHTS = {
    "general": {
        "textrank": 0.30,
        "centroid": 0.25,
        "position": 0.15,
        "keyword":  0.20,
        "length":   0.10,
    },
    "news": {
        # Lead bias: position matters most
        "textrank": 0.25,
        "centroid": 0.20,
        "position": 0.30,
        "keyword":  0.20,
        "length":   0.05,
    },
    "academic": {
        # Structure and topic coverage are key
        "textrank": 0.35,
        "centroid": 0.30,
        "position": 0.10,
        "keyword":  0.20,
        "length":   0.05,
    },
    "legal": {
        # Definitions and named entities matter most
        "textrank": 0.25,
        "centroid": 0.25,
        "position": 0.10,
        "keyword":  0.35,
        "length":   0.05,
    },
}

In [11]:
print("\n── 3c. Length Penalty Scores ──")
ln = length_penalty_scores(sentences)
for i, (s, sc) in enumerate(zip(sentences, ln)):
    wc  = len(s.split())
    bar = "█" * int(sc * 20)
    print(f"  S{i+1}  {sc:.3f}  {bar:<20}  ({wc:>2} words)  {s[:45]}...")


── 3c. Length Penalty Scores ──
  S1  1.000  ████████████████████  (33 words)  The Ultimate Productivity Hack is Saying No -...
  S2  1.000  ████████████████████  (10 words)  Not doing something will always be faster tha...
  S3  1.000  ████████████████████  (30 words)  This statement reminds me of the old computer...
  S4  1.000  ████████████████████  (16 words)  For example, there is no meeting that goes fa...
  S5  1.000  ████████████████████  (28 words)  This is not to say you should never attend an...
  S6  1.000  ████████████████████  (11 words)  There are many meetings held that don't need ...
  S7  1.000  ████████████████████  (11 words)  There is a lot of code written that could be ...
  S8  1.000  ████████████████████  (28 words)  How often do people ask you to do something a...
  S9  1.000  ████████████████████  (21 words)  We become frustrated by our obligations even ...
  S10  0.800  ████████████████      ( 8 words)  2 It's worth asking if things are necessary....
  S11  

### 5. Combine Scores

In [12]:
# 3e. SCORE COMBINATION

def combine_features(sentences: list[str],
                     textrank_scores_arr: np.ndarray,
                     centroid_scores_arr: np.ndarray,
                     domain: str = "general") -> dict:
    """
    Combine all 5 signals into a single final score per sentence.
 
    Parameters
    ----------
    sentences           : list of cleaned sentence strings (from Step 1)
    textrank_scores_arr : TextRank scores from Step 2
    centroid_scores_arr : centroid scores from Step 2
    domain              : "general" | "news" | "academic" | "legal"
 
    Returns
    -------
    dict containing:
        textrank  : TextRank signal
        centroid  : centroid signal
        position  : position signal
        keyword   : keyword overlap signal
        length    : length penalty signal
        combined  : final weighted score  ← pass this to Step 4
        keywords  : the keyword set used
        domain    : domain used
        weights   : weights used
    """
    weights   = DOMAIN_WEIGHTS.get(domain, DOMAIN_WEIGHTS["general"])
    full_text = " ".join(sentences)
 
    # Compute the 3 new features
    pos_scores = position_scores(sentences)
    keywords   = extract_keywords(full_text, top_n=15)
    kw_scores  = keyword_overlap_scores(sentences, keywords)
    len_scores = length_penalty_scores(sentences)
 
    # Weighted combination
    combined = (
        weights["textrank"] * textrank_scores_arr +
        weights["centroid"] * centroid_scores_arr +
        weights["position"] * pos_scores +
        weights["keyword"]  * kw_scores +
        weights["length"]   * len_scores
    )
 
    # Re-normalise combined to [0, 1]
    c_min, c_max = combined.min(), combined.max()
    if c_max > c_min:
        combined = (combined - c_min) / (c_max - c_min)
 
    return {
        "textrank": textrank_scores_arr,
        "centroid": centroid_scores_arr,
        "position": pos_scores,
        "keyword":  kw_scores,
        "length":   len_scores,
        "combined": combined,
        "keywords": keywords,
        "domain":   domain,
        "weights":  weights,
    }

In [13]:
for domain in ["general", "news", "academic", "legal"]:
    result = combine_features(
    sentences,
        combined["textrank"],
        combined["centroid"],
        domain=domain
    )
    print(f"\n── 3d+3e. Combined Scores — domain='{domain}' ──")
    print(f"  Weights: { {k: f'{v:.2f}' for k, v in result['weights'].items()} }")
    print(f"\n  {'#':<3} {'TR':>6} {'Cen':>6} {'Pos':>6} {'KW':>6} {'Len':>6} {'FINAL':>7}  Sentence")
    print("  " + "-" * 82)
    for i, s in enumerate(sentences):
        print(f"  {i+1:<3} "
                f"{result['textrank'][i]:>6.3f} "
                f"{result['centroid'][i]:>6.3f} "
                f"{result['position'][i]:>6.3f} "
                f"{result['keyword'][i]:>6.3f} "
                f"{result['length'][i]:>6.3f} "
                f"{result['combined'][i]:>7.3f}  "
                f"{s[:48]}...")


── 3d+3e. Combined Scores — domain='general' ──
  Weights: {'textrank': '0.30', 'centroid': '0.25', 'position': '0.15', 'keyword': '0.20', 'length': '0.10'}

  #       TR    Cen    Pos     KW    Len   FINAL  Sentence
  ----------------------------------------------------------------------------------
  1    0.617  0.343  1.000  0.500  1.000   0.699  The Ultimate Productivity Hack is Saying No - Ja...
  2    0.739  0.161  0.985  0.167  1.000   0.594  Not doing something will always be faster than d...
  3    0.749  0.229  0.969  0.333  1.000   0.663  This statement reminds me of the old computer pr...
  4    0.681  0.149  0.954  0.000  1.000   0.516  For example, there is no meeting that goes faste...
  5    0.759  0.787  0.939  0.500  1.000   0.893  This is not to say you should never attend anoth...
  6    0.530  0.182  0.923  0.333  1.000   0.549  There are many meetings held that don't need to ...
  7    0.236  0.048  0.908  0.000  1.000   0.293  There is a lot of code written that

In [16]:
print("\n── How domain changes top-3 ranking ──")
for domain in ["general", "news", "academic", "legal"]:
    result  = combine_features(sentences, combined["textrank"], combined["centroid"], domain=domain)
    top3    = np.argsort(result["combined"])[::-1][:3]
    print(f"\n  {domain.upper()}")
    for rank, idx in enumerate(top3, 1):
        print(f"    #{rank} (score={result['combined'][idx]:.3f})  {sentences[idx][:65]}...")


── How domain changes top-3 ranking ──

  GENERAL
    #1 (score=1.000)  I like how the economist Tim Harford put it, “Every time we say y...
    #2 (score=0.905)  Saying yes costs you time in the future....
    #3 (score=0.893)  This is not to say you should never attend another meeting, but t...

  NEWS
    #1 (score=1.000)  I like how the economist Tim Harford put it, “Every time we say y...
    #2 (score=0.966)  This is not to say you should never attend another meeting, but t...
    #3 (score=0.913)  But if the benefits of saying no are so obvious, then why do we s...

  ACADEMIC
    #1 (score=1.000)  I like how the economist Tim Harford put it, “Every time we say y...
    #2 (score=0.927)  Saying yes costs you time in the future....
    #3 (score=0.864)  When you say yes, you are saying no to every other option....

  LEGAL
    #1 (score=1.000)  I like how the economist Tim Harford put it, “Every time we say y...
    #2 (score=0.824)  Saying yes costs you time in the future....
 